In [ ]:
from pathlib import Path
import sys
import sklearn, numpy, pandas

REPO_ROOT = next(
    (candidate for candidate in [Path.cwd(), *Path.cwd().parents]
     if (candidate / "ai-models" / "data" / "dataset.zip").exists()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("KhÃƒÂ´ng tÃƒÂ¬m thÃ¡ÂºÂ¥y ai-models/data/dataset.zip trong repo hiÃ¡Â»â€¡n tÃ¡ÂºÂ¡i")

sys.path.insert(0, str(REPO_ROOT / "ai-models" / "src"))
print("sklearn", sklearn.__version__, "| numpy", numpy.__version__, "| pandas", pandas.__version__)
print("Repo:", REPO_ROOT)

In [ ]:
import warnings; warnings.filterwarnings("ignore")
from preprocess import load_data, split, ROOT

X, y = load_data()
X_train, X_test, y_train, y_test = split(X, y)
print("Train:", X_train.shape, "| Test:", X_test.shape)
print("TÃ¡Â»â€° lÃ¡Â»â€¡ ÃƒÂ¡c tÃƒÂ­nh - train:", round(y_train.mean(), 3), "| test:", round(y_test.mean(), 3))

In [ ]:
from train import run_experiments

results, fitted = run_experiments(X_train, X_test, y_train, y_test)
results

In [ ]:
cols = ["model", "best_params", "cv_f1_mean", "cv_f1_std", "fit_seconds",
        "predict_ms_per_sample", "size_kb", "train_f1",
        "test_recall", "test_precision", "test_f1", "test_roc_auc"]
print(results[cols].to_csv(index=False))

In [ ]:
cols = ["model", "test_recall", "test_precision", "test_f1", "test_roc_auc",
        "train_f1", "predict_ms_per_sample", "size_kb"]
results[cols].sort_values("test_recall", ascending=False)

In [ ]:
results[["model", "best_params", "cv_f1_mean", "cv_f1_std", "fit_seconds"]]

In [ ]:
print("NÃ¡ÂºÂ¿u luÃƒÂ´n Ã„â€˜oÃƒÂ¡n 'lÃƒÂ nh tÃƒÂ­nh': Accuracy =", round(1 - y_test.mean(), 3), "| Recall ÃƒÂ¡c tÃƒÂ­nh = 0")

In [ ]:
results.to_csv(ROOT / "docs" / "model_comparison.csv", index=False)
print("Ã„ÂÃƒÂ£ lÃ†Â°u bÃ¡ÂºÂ£ng so sÃƒÂ¡nh:", ROOT / "docs" / "model_comparison.csv")

## HuÃ¡ÂºÂ¥n luyÃ¡Â»â€¡n vÃƒÂ  tinh chÃ¡Â»â€°nh 5 model

**CÃƒÂ¡ch lÃƒÂ m chung (Ã„â€˜Ã¡Â»Æ’ so sÃƒÂ¡nh cÃƒÂ´ng bÃ¡ÂºÂ±ng):**
- CÃƒÂ¹ng bÃ¡Â»â„¢ dÃ¡Â»Â¯ liÃ¡Â»â€¡u, cÃƒÂ¹ng cÃƒÂ¡ch chia: 80% train / 20% test (455 / 114 mÃ¡ÂºÂ«u), `stratify=y`, `random_state=42`.
- MÃ¡Â»â€”i model nÃ¡ÂºÂ±m trong mÃ¡Â»â„¢t `Pipeline` gÃ¡Â»â€œm imputer, StandardScaler vÃƒÂ  model, nÃƒÂªn cÃƒÂ¡c bÃ†Â°Ã¡Â»â€ºc tiÃ¡Â»Ân xÃ¡Â»Â­ lÃƒÂ½ chÃ¡Â»â€° Ã„â€˜Ã†Â°Ã¡Â»Â£c fit trÃƒÂªn tÃ¡ÂºÂ­p train (khÃƒÂ´ng rÃƒÂ² rÃ¡Â»â€° dÃ¡Â»Â¯ liÃ¡Â»â€¡u).
- Tinh chÃ¡Â»â€°nh siÃƒÂªu tham sÃ¡Â»â€˜ bÃ¡ÂºÂ±ng `GridSearchCV` vÃ¡Â»â€ºi 5-fold cross-validation **chÃ¡Â»â€° trÃƒÂªn tÃ¡ÂºÂ­p train**, tÃ¡Â»â€˜i Ã†Â°u theo F1 cÃ¡Â»Â§a lÃ¡Â»â€ºp ÃƒÂ¡c tÃƒÂ­nh.
- TÃ¡ÂºÂ­p test chÃ¡Â»â€° dÃƒÂ¹ng **mÃ¡Â»â„¢t lÃ¡ÂºÂ§n** Ã„â€˜Ã¡Â»Æ’ lÃ¡ÂºÂ¥y kÃ¡ÂºÂ¿t quÃ¡ÂºÂ£ cuÃ¡Â»â€˜i.
- Metric trÃ¡Â»Âng tÃƒÂ¢m: **Recall cÃ¡Â»Â§a lÃ¡Â»â€ºp ÃƒÂ¡c tÃƒÂ­nh**, vÃƒÂ¬ bÃ¡Â»Â sÃƒÂ³t ca ung thÃ†Â° (FN) nguy hiÃ¡Â»Æ’m hÃ†Â¡n bÃƒÂ¡o nhÃ¡ÂºÂ§m (FP). NgoÃƒÂ i ra dÃƒÂ¹ng Precision, F1, ROC-AUC.
- Ghi chÃƒÂº: "thÃ¡Â»Âi gian huÃ¡ÂºÂ¥n luyÃ¡Â»â€¡n" bÃƒÂªn dÃ†Â°Ã¡Â»â€ºi lÃƒÂ  tÃ¡Â»â€¢ng thÃ¡Â»Âi gian chÃ¡ÂºÂ¡y toÃƒÂ n bÃ¡Â»â„¢ `GridSearchCV` (gÃ¡Â»â€œm mÃ¡Â»Âi tÃ¡Â»â€¢ hÃ¡Â»Â£p tham sÃ¡Â»â€˜ Ãƒâ€” 5 fold), khÃƒÂ´ng phÃ¡ÂºÂ£i mÃ¡Â»â„¢t lÃ¡ÂºÂ§n fit. "ThÃ¡Â»Âi gian dÃ¡Â»Â± Ã„â€˜oÃƒÂ¡n" lÃƒÂ  trung bÃƒÂ¬nh trÃƒÂªn mÃ¡Â»â€”i mÃ¡ÂºÂ«u cÃ¡Â»Â§a tÃ¡ÂºÂ­p test.

### 1. Logistic Regression (baseline)
- **NguyÃƒÂªn lÃƒÂ½:** tÃƒÂ­nh tÃ¡Â»â€¢ hÃ¡Â»Â£p tuyÃ¡ÂºÂ¿n tÃƒÂ­nh cÃ¡Â»Â§a 30 Ã„â€˜Ã¡ÂºÂ·c trÃ†Â°ng rÃ¡Â»â€œi Ã„â€˜Ã†Â°a qua hÃƒÂ m sigmoid Ã„â€˜Ã¡Â»Æ’ ra xÃƒÂ¡c suÃ¡ÂºÂ¥t ÃƒÂ¡c tÃƒÂ­nh; xÃƒÂ¡c suÃ¡ÂºÂ¥t Ã¢â€°Â¥ 0.5 thÃƒÂ¬ dÃ¡Â»Â± Ã„â€˜oÃƒÂ¡n ÃƒÂ¡c tÃƒÂ­nh. Ranh giÃ¡Â»â€ºi quyÃ¡ÂºÂ¿t Ã„â€˜Ã¡Â»â€¹nh lÃƒÂ  mÃ¡Â»â„¢t siÃƒÂªu phÃ¡ÂºÂ³ng.
- **VÃƒÂ¬ sao lÃƒÂ m baseline:** Ã„â€˜Ã†Â¡n giÃ¡ÂºÂ£n, nhanh, dÃ¡Â»â€¦ giÃ¡ÂºÂ£i thÃƒÂ­ch; lÃƒÂ  mÃ¡Â»â€˜c Ã„â€˜Ã¡Â»Æ’ biÃ¡ÂºÂ¿t cÃƒÂ¡c model phÃ¡Â»Â©c tÃ¡ÂºÂ¡p hÃ†Â¡n cÃƒÂ³ thÃ¡Â»Â±c sÃ¡Â»Â± Ã„â€˜ÃƒÂ¡ng giÃƒÂ¡ khÃƒÂ´ng.
- **SiÃƒÂªu tham sÃ¡Â»â€˜:** `C` (nghÃ¡Â»â€¹ch Ã„â€˜Ã¡ÂºÂ£o Ã„â€˜Ã¡Â»â„¢ mÃ¡ÂºÂ¡nh cÃ¡Â»Â§a Ã„â€˜iÃ¡Â»Âu chuÃ¡ÂºÂ©n L2). Ã„ÂÃƒÂ£ thÃ¡Â»Â­ C Ã¢Ë†Ë† {0.01, 0.1, 1, 10, 100}. **ChÃ¡Â»Ân C = 0.1**, tÃ¡Â»Â©c Ã„â€˜iÃ¡Â»Âu chuÃ¡ÂºÂ©n khÃƒÂ¡ mÃ¡ÂºÂ¡nh. C nhÃ¡Â»Â thÃƒÂ¬ model Ã„â€˜Ã†Â¡n giÃ¡ÂºÂ£n, dÃ¡Â»â€¦ underfit; C lÃ¡Â»â€ºn thÃƒÂ¬ Ã„â€˜iÃ¡Â»Âu chuÃ¡ÂºÂ©n yÃ¡ÂºÂ¿u, dÃ¡Â»â€¦ overfit. `class_weight="balanced"` Ã„â€˜Ã¡Â»Æ’ bÃƒÂ¹ lÃ¡Â»â€¡ch lÃ¡Â»â€ºp nhÃ¡ÂºÂ¹ (khoÃ¡ÂºÂ£ng 63% lÃƒÂ nh / 37% ÃƒÂ¡c).
- **ThÃ¡Â»Âi gian huÃ¡ÂºÂ¥n luyÃ¡Â»â€¡n (cÃ¡ÂºÂ£ GridSearch):** 2.4 giÃƒÂ¢y | **DÃ¡Â»Â± Ã„â€˜oÃƒÂ¡n:** 0.029 ms/mÃ¡ÂºÂ«u | **KÃƒÂ­ch thÃ†Â°Ã¡Â»â€ºc file:** 2.4 KB.
- **Cross-validation:** F1 trung bÃƒÂ¬nh 0.9671, Ã„â€˜Ã¡Â»â„¢ lÃ¡Â»â€¡ch chuÃ¡ÂºÂ©n 0.0112.

### 2. KNN
- **NguyÃƒÂªn lÃƒÂ½:** khÃƒÂ´ng hÃ¡Â»Âc tham sÃ¡Â»â€˜. VÃ¡Â»â€ºi mÃ¡ÂºÂ«u mÃ¡Â»â€ºi, tÃƒÂ¬m K mÃ¡ÂºÂ«u train gÃ¡ÂºÂ§n nhÃ¡ÂºÂ¥t (khoÃ¡ÂºÂ£ng cÃƒÂ¡ch Euclid trÃƒÂªn dÃ¡Â»Â¯ liÃ¡Â»â€¡u Ã„â€˜ÃƒÂ£ chuÃ¡ÂºÂ©n hÃƒÂ³a) rÃ¡Â»â€œi bÃ¡Â»Â phiÃ¡ÂºÂ¿u theo Ã„â€˜a sÃ¡Â»â€˜.
- **VÃƒÂ¬ sao cÃ¡ÂºÂ§n chuÃ¡ÂºÂ©n hÃƒÂ³a:** khoÃ¡ÂºÂ£ng cÃƒÂ¡ch bÃ¡Â»â€¹ chi phÃ¡Â»â€˜i bÃ¡Â»Å¸i cÃ¡Â»â„¢t cÃƒÂ³ thang Ã„â€˜o lÃ¡Â»â€ºn (vÃƒÂ­ dÃ¡Â»Â¥ `area` hÃƒÂ ng trÃ„Æ’m so vÃ¡Â»â€ºi `smoothness` dÃ†Â°Ã¡Â»â€ºi 1).
- **SiÃƒÂªu tham sÃ¡Â»â€˜:** `n_neighbors` Ã¢Ë†Ë† {3, 5, 7, 9, 11, 15}, `weights` Ã¢Ë†Ë† {uniform, distance}. **ChÃ¡Â»Ân K = 3, weights = uniform.** K nhÃ¡Â»Â thÃƒÂ¬ biÃƒÂªn phÃ¡Â»Â©c tÃ¡ÂºÂ¡p, nhÃ¡ÂºÂ¡y nhiÃ¡Â»â€¦u, dÃ¡Â»â€¦ overfit; K lÃ¡Â»â€ºn thÃƒÂ¬ biÃƒÂªn mÃ†Â°Ã¡Â»Â£t, dÃ¡Â»â€¦ underfit. K = 3 lÃƒÂ  giÃƒÂ¡ trÃ¡Â»â€¹ nhÃ¡Â»Â nhÃ¡ÂºÂ¥t trong lÃ†Â°Ã¡Â»â€ºi, cho thÃ¡ÂºÂ¥y model bÃƒÂ¡m sÃƒÂ¡t dÃ¡Â»Â¯ liÃ¡Â»â€¡u train vÃƒÂ  cÃƒÂ³ xu hÃ†Â°Ã¡Â»â€ºng overfit (Train F1 0.979 so vÃ¡Â»â€ºi Test F1 0.911).
- **ThÃ¡Â»Âi gian huÃ¡ÂºÂ¥n luyÃ¡Â»â€¡n (cÃ¡ÂºÂ£ GridSearch):** 0.8 giÃƒÂ¢y | **DÃ¡Â»Â± Ã„â€˜oÃƒÂ¡n:** 0.036 ms/mÃ¡ÂºÂ«u | **KÃƒÂ­ch thÃ†Â°Ã¡Â»â€ºc file:** 102.3 KB (lÃ¡Â»â€ºn nhÃ¡ÂºÂ¥t, vÃƒÂ¬ lÃ†Â°u cÃ¡ÂºÂ£ dÃ¡Â»Â¯ liÃ¡Â»â€¡u train).
- **Cross-validation:** F1 trung bÃƒÂ¬nh 0.9633, Ã„â€˜Ã¡Â»â„¢ lÃ¡Â»â€¡ch chuÃ¡ÂºÂ©n 0.0250 (kÃƒÂ©m Ã¡Â»â€¢n Ã„â€˜Ã¡Â»â€¹nh hÃ†Â¡n Logistic vÃƒÂ  SVM).

### 3. SVM (kernel RBF)
- **NguyÃƒÂªn lÃƒÂ½:** tÃƒÂ¬m ranh giÃ¡Â»â€ºi cÃƒÂ³ **lÃ¡Â»Â lÃ¡Â»â€ºn nhÃ¡ÂºÂ¥t** giÃ¡Â»Â¯a hai lÃ¡Â»â€ºp. Kernel RBF ÃƒÂ¡nh xÃ¡ÂºÂ¡ ngÃ¡ÂºÂ§m dÃ¡Â»Â¯ liÃ¡Â»â€¡u sang khÃƒÂ´ng gian nhiÃ¡Â»Âu chiÃ¡Â»Âu Ã„â€˜Ã¡Â»Æ’ tÃƒÂ¡ch Ã„â€˜Ã†Â°Ã¡Â»Â£c ranh giÃ¡Â»â€ºi phi tuyÃ¡ÂºÂ¿n.
- **SiÃƒÂªu tham sÃ¡Â»â€˜:** `C` Ã¢Ë†Ë† {0.1, 1, 10, 100} (Ã„â€˜ÃƒÂ¡nh Ã„â€˜Ã¡Â»â€¢i giÃ¡Â»Â¯a lÃ¡Â»Â rÃ¡Â»â„¢ng vÃƒÂ  sÃ¡Â»â€˜ Ã„â€˜iÃ¡Â»Æ’m bÃ¡Â»â€¹ phÃƒÂ¢n loÃ¡ÂºÂ¡i sai), `gamma` Ã¢Ë†Ë† {scale, 0.01, 0.1} (Ã„â€˜Ã¡Â»â„¢ "cong" cÃ¡Â»Â§a biÃƒÂªn; gamma lÃ¡Â»â€ºn thÃƒÂ¬ mÃ¡Â»â€”i Ã„â€˜iÃ¡Â»Æ’m Ã¡ÂºÂ£nh hÃ†Â°Ã¡Â»Å¸ng hÃ¡ÂºÂ¹p, biÃƒÂªn uÃ¡Â»â€˜n lÃ†Â°Ã¡Â»Â£n, dÃ¡Â»â€¦ overfit). **ChÃ¡Â»Ân C = 10, gamma = 0.01**: gamma nhÃ¡Â»Â cho biÃƒÂªn mÃ†Â°Ã¡Â»Â£t, C = 10 cho phÃƒÂ©p mÃ¡Â»â„¢t sÃ¡Â»â€˜ sai sÃ¡Â»â€˜ Ã„â€˜Ã¡Â»Æ’ lÃ¡Â»Â Ã„â€˜Ã¡Â»Â§ rÃ¡Â»â„¢ng.
- `probability=True` Ã„â€˜Ã¡Â»Æ’ cÃƒÂ³ xÃƒÂ¡c suÃ¡ÂºÂ¥t phÃ¡Â»Â¥c vÃ¡Â»Â¥ ROC-AUC vÃƒÂ  hiÃ¡Â»Æ’n thÃ¡Â»â€¹ trÃƒÂªn app; `class_weight="balanced"`.
- **ThÃ¡Â»Âi gian huÃ¡ÂºÂ¥n luyÃ¡Â»â€¡n (cÃ¡ÂºÂ£ GridSearch):** 1.9 giÃƒÂ¢y | **DÃ¡Â»Â± Ã„â€˜oÃƒÂ¡n:** 0.028 ms/mÃ¡ÂºÂ«u | **KÃƒÂ­ch thÃ†Â°Ã¡Â»â€ºc file:** 16.6 KB.
- **Cross-validation:** F1 trung bÃƒÂ¬nh 0.9730 (cao nhÃ¡ÂºÂ¥t trong 5 model), Ã„â€˜Ã¡Â»â„¢ lÃ¡Â»â€¡ch chuÃ¡ÂºÂ©n 0.0116.

### 4. Decision Tree
- **NguyÃƒÂªn lÃƒÂ½:** chia dÃ¡Â»Â¯ liÃ¡Â»â€¡u liÃƒÂªn tiÃ¡ÂºÂ¿p bÃ¡ÂºÂ±ng cÃƒÂ¡c Ã„â€˜iÃ¡Â»Âu kiÃ¡Â»â€¡n dÃ¡ÂºÂ¡ng "Ã„â€˜Ã¡ÂºÂ·c trÃ†Â°ng Ã¢â€°Â¤ ngÃ†Â°Ã¡Â»Â¡ng?". MÃ¡Â»â€”i lÃ¡ÂºÂ§n chia chÃ¡Â»Ân Ã„â€˜iÃ¡Â»Âu kiÃ¡Â»â€¡n lÃƒÂ m giÃ¡ÂºÂ£m Ã„â€˜Ã¡Â»â„¢ tÃ¡ÂºÂ¡p chÃ¡ÂºÂ¥t (Gini hoÃ¡ÂºÂ·c Entropy) nhiÃ¡Â»Âu nhÃ¡ÂºÂ¥t.
- **Ã†Â¯u Ã„â€˜iÃ¡Â»Æ’m:** dÃ¡Â»â€¦ giÃ¡ÂºÂ£i thÃƒÂ­ch (cÃƒÂ³ thÃ¡Â»Æ’ vÃ¡ÂºÂ½ thÃƒÂ nh cÃƒÂ¢y), khÃƒÂ´ng cÃ¡ÂºÂ§n chuÃ¡ÂºÂ©n hÃƒÂ³a. **NhÃ†Â°Ã¡Â»Â£c Ã„â€˜iÃ¡Â»Æ’m:** cÃƒÂ¢y sÃƒÂ¢u rÃ¡ÂºÂ¥t dÃ¡Â»â€¦ overfit.
- **SiÃƒÂªu tham sÃ¡Â»â€˜:** `criterion` Ã¢Ë†Ë† {gini, entropy}, `max_depth` Ã¢Ë†Ë† {3, 5, 7, 10, None}, `min_samples_leaf` Ã¢Ë†Ë† {1, 2, 5}. GiÃ¡Â»â€ºi hÃ¡ÂºÂ¡n Ã„â€˜Ã¡Â»â„¢ sÃƒÂ¢u vÃƒÂ  sÃ¡Â»â€˜ mÃ¡ÂºÂ«u tÃ¡Â»â€˜i thiÃ¡Â»Æ’u Ã¡Â»Å¸ lÃƒÂ¡ chÃƒÂ­nh lÃƒÂ  cÃƒÂ¡ch hÃ¡ÂºÂ¡n chÃ¡ÂºÂ¿ Ã„â€˜Ã¡Â»â„¢ phÃ¡Â»Â©c tÃ¡ÂºÂ¡p cÃ¡Â»Â§a cÃƒÂ¢y (pruning). **ChÃ¡Â»Ân criterion = gini, max_depth = 5, min_samples_leaf = 5.**
- **ThÃ¡Â»Âi gian huÃ¡ÂºÂ¥n luyÃ¡Â»â€¡n (cÃ¡ÂºÂ£ GridSearch):** 3.2 giÃƒÂ¢y (lÃ†Â°Ã¡Â»â€ºi cÃƒÂ³ 30 tÃ¡Â»â€¢ hÃ¡Â»Â£p) | **DÃ¡Â»Â± Ã„â€˜oÃƒÂ¡n:** 0.032 ms/mÃ¡ÂºÂ«u | **KÃƒÂ­ch thÃ†Â°Ã¡Â»â€ºc file:** 3.2 KB.
- **Cross-validation:** F1 trung bÃƒÂ¬nh 0.9378 (thÃ¡ÂºÂ¥p nhÃ¡ÂºÂ¥t), Ã„â€˜Ã¡Â»â„¢ lÃ¡Â»â€¡ch chuÃ¡ÂºÂ©n 0.0297 (kÃƒÂ©m Ã¡Â»â€¢n Ã„â€˜Ã¡Â»â€¹nh nhÃ¡ÂºÂ¥t).

### 5. Random Forest
- **NguyÃƒÂªn lÃƒÂ½:** huÃ¡ÂºÂ¥n luyÃ¡Â»â€¡n nhiÃ¡Â»Âu cÃƒÂ¢y, mÃ¡Â»â€”i cÃƒÂ¢y hÃ¡Â»Âc trÃƒÂªn mÃ¡Â»â„¢t mÃ¡ÂºÂ«u bootstrap cÃ¡Â»Â§a dÃ¡Â»Â¯ liÃ¡Â»â€¡u vÃƒÂ  chÃ¡Â»â€° xÃƒÂ©t ngÃ¡ÂºÂ«u nhiÃƒÂªn mÃ¡Â»â„¢t tÃ¡ÂºÂ­p Ã„â€˜Ã¡ÂºÂ·c trÃ†Â°ng Ã¡Â»Å¸ mÃ¡Â»â€”i lÃ¡ÂºÂ§n chia; kÃ¡ÂºÂ¿t quÃ¡ÂºÂ£ bÃ¡Â»Â phiÃ¡ÂºÂ¿u theo Ã„â€˜a sÃ¡Â»â€˜ (bagging). Trung bÃƒÂ¬nh nhiÃ¡Â»Âu cÃƒÂ¢y lÃƒÂ m giÃ¡ÂºÂ£m phÃ†Â°Ã†Â¡ng sai so vÃ¡Â»â€ºi mÃ¡Â»â„¢t cÃƒÂ¢y Ã„â€˜Ã†Â¡n.
- **SiÃƒÂªu tham sÃ¡Â»â€˜:** `n_estimators` Ã¢Ë†Ë† {100, 200, 300}, `max_depth` Ã¢Ë†Ë† {None, 5, 10}, `min_samples_leaf` Ã¢Ë†Ë† {1, 2}. **ChÃ¡Â»Ân n_estimators = 100, max_depth = None, min_samples_leaf = 1.** RÃ¡Â»Â«ng Ã„â€˜ÃƒÂ£ Ã¡Â»â€¢n Ã„â€˜Ã¡Â»â€¹nh vÃ¡Â»â€ºi 100 cÃƒÂ¢y nÃƒÂªn thÃƒÂªm cÃƒÂ¢y khÃƒÂ´ng cÃ¡ÂºÂ£i thiÃ¡Â»â€¡n thÃƒÂªm.
- **ThÃ¡Â»Âi gian huÃ¡ÂºÂ¥n luyÃ¡Â»â€¡n (cÃ¡ÂºÂ£ GridSearch):** 44.9 giÃƒÂ¢y (lÃƒÂ¢u nhÃ¡ÂºÂ¥t, vÃƒÂ¬ 18 tÃ¡Â»â€¢ hÃ¡Â»Â£p Ãƒâ€” 5 fold, mÃ¡Â»â€”i lÃ¡ÂºÂ§n fit 100 Ã„â€˜Ã¡ÂºÂ¿n 300 cÃƒÂ¢y) | **DÃ¡Â»Â± Ã„â€˜oÃƒÂ¡n:** 0.33 ms/mÃ¡ÂºÂ«u (chÃ¡ÂºÂ­m nhÃ¡ÂºÂ¥t) | **KÃƒÂ­ch thÃ†Â°Ã¡Â»â€ºc file:** 89.4 KB.
- **Cross-validation:** F1 trung bÃƒÂ¬nh 0.9525, Ã„â€˜Ã¡Â»â„¢ lÃ¡Â»â€¡ch chuÃ¡ÂºÂ©n 0.0175.

### NhÃ¡ÂºÂ­n xÃƒÂ©t sÃ†Â¡ bÃ¡Â»â„¢ tÃ¡Â»Â« bÃ¡ÂºÂ£ng kÃ¡ÂºÂ¿t quÃ¡ÂºÂ£

| Model | Train F1 | Test F1 | ChÃƒÂªnh (train Ã¢Ë†â€™ test) | Test Recall | Test Precision |
|---|---|---|---|---|---|
| Logistic Regression (baseline) | 0.9794 | 0.9880 | Ã¢Ë†â€™0.009 | 0.9762 | 1.0000 |
| KNN | 0.9790 | 0.9114 | 0.068 | 0.8571 | 0.9730 |
| SVM (RBF) | 0.9851 | 0.9880 | Ã¢Ë†â€™0.003 | 0.9762 | 1.0000 |
| Decision Tree | 0.9510 | 0.8642 | 0.087 | 0.8333 | 0.8974 |
| Random Forest | 1.0000 | 0.9630 | 0.037 | 0.9286 | 1.0000 |

- **Recall test cao nhÃ¡ÂºÂ¥t:** Logistic Regression vÃƒÂ  SVM (cÃƒÂ¹ng 0.9762).
- **Ã¡Â»â€n Ã„â€˜Ã¡Â»â€¹nh nhÃ¡ÂºÂ¥t (`cv_f1_std` nhÃ¡Â»Â nhÃ¡ÂºÂ¥t):** Logistic Regression (0.0112), sÃƒÂ¡t SVM (0.0116).
- **CÃƒÂ³ dÃ¡ÂºÂ¥u hiÃ¡Â»â€¡u overfit:** Decision Tree (chÃƒÂªnh 0.087), KNN (chÃƒÂªnh 0.068), Random Forest (Train F1 = 1.0 nhÃ†Â°ng Test F1 = 0.963). Logistic Regression vÃƒÂ  SVM cÃƒÂ³ Test F1 khÃƒÂ´ng thÃ¡ÂºÂ¥p hÃ†Â¡n Train F1, tÃ¡Â»Â©c khÃƒÂ´ng overfit. ViÃ¡Â»â€¡c Test F1 nhÃ¡Â»â€°nh hÃ†Â¡n Train F1 mÃ¡Â»â„¢t chÃƒÂºt lÃƒÂ  do tÃ¡ÂºÂ­p test nhÃ¡Â»Â (114 mÃ¡ÂºÂ«u), khÃƒÂ´ng phÃ¡ÂºÂ£i dÃ¡ÂºÂ¥u hiÃ¡Â»â€¡u bÃ¡ÂºÂ¥t thÃ†Â°Ã¡Â»Âng.
- **So vÃ¡Â»â€ºi baseline Logistic Regression (Test F1 = 0.988):** khÃƒÂ´ng model nÃƒÂ o vÃ†Â°Ã¡Â»Â£t baseline trÃƒÂªn tÃ¡ÂºÂ­p test; SVM chÃ¡Â»â€° ngang bÃ¡ÂºÂ±ng. Ã„ÂiÃ¡Â»Âu nÃƒÂ y cho thÃ¡ÂºÂ¥y ranh giÃ¡Â»â€ºi giÃ¡Â»Â¯a hai lÃ¡Â»â€ºp gÃ¡ÂºÂ§n nhÃ†Â° tuyÃ¡ÂºÂ¿n tÃƒÂ­nh, khÃ¡Â»â€ºp vÃ¡Â»â€ºi pairplot Ã¡Â»Å¸ phÃ¡ÂºÂ§n EDA (hai lÃ¡Â»â€ºp tÃƒÂ¡ch khÃƒÂ¡ rÃƒÂµ), nÃƒÂªn model Ã„â€˜Ã†Â¡n giÃ¡ÂºÂ£n Ã„â€˜ÃƒÂ£ Ã„â€˜Ã¡Â»Â§ tÃ¡Â»â€˜t.